# Annotating EMISSOR scenario

EMISSOR is an open framework for representing multimodal interaction as a sequence of signals that are grounded in time and place. EMISSOR creates a JSON file with the meta data on each signal, i.e. image, audio or text. It provides identifiers for the context, the signals and their grounding in time such as the start and end of a signal and place, the location where a signal is recorded. In addition to the signals, EMISSOR also represents interpretations of these signals as annotations. In this notebook, we will show how signals are annotated. We will only discusse text signals. 
Please consult the [Leolani Github](https://github.com/leolani) for annotations of other signals such as audio and images.

We will consider four annotations:

1. Dialogue acts
2. Emotions and sentiments
3. Tokens, their part-of-speech and named entities
4. The likelihood of utterances in a sequence

These annotations will give an impression of the quality of the interaction as communication.

Annotations are attached to signals as part of a so-called ```mention```. A mention is a data element that relates a specification of a segment from the signal with an annotation. For example, a specific segment of an image, defined by a boudning box, can be interpreted as a human face in the annotation. Similarly, a specific part of an utterance in a text signal, defined by the offset position and length, can be interpreted as the mentioning of a person by his name. Below is an example of a mention where we focus on an annotation element:

```
      {
        "@context": {...},
        "@type": "Mention",
        "segment": [
          {...}
        ],
        "annotations": [
          {
            "timestamp": 1732101081681,
            "@context": {...},
            "value": {
              "type": "GO",
              "value": "neutral",
              "confidence": 0.6464349627494812,
              "_py_type": "emissor.representation.util-JSON"
            },
            "@type": "Annotation",
            "source": "GO",
            "type": "python-type:cltl.emotion_extraction.api.Emotion",
            "_py_type": "emissor.representation.scenario-Annotation"
          },
        ],
        "id": "0116513d-dbec-4a5c-a54f-d29f7732ec7a"
      },
```

The annotations are grouped with a segment to form a mention. The segment is hidden here but the annotation element is fully shown.
The annotation has a timestamp, a type, a source and a value. The timestamp indicates when the an annotation was done. The type indicates which Python object it is and the source which tool created this annotation. The most important part is the value that gives the actual label as a value, the type as the category for the value and the confidence for the trust in the value to be correct.

In [1]:
### Importing the annotators from installed pip modules
from cltl.dialogue_act_classification.add_dialogue_acts_to_emissor import DialogueActAnnotator
from cltl.emotion_extraction.add_emotions_to_emissor import EmotionAnnotator
from cltl.nlp.add_nlp_to_emissor import NLPAnnotator
from cltl.dialogue_evaluation.add_likelihood_to_emissor import LikelihoodAnnotator
import cltl.dialogue_evaluation.utils as util

### Importing the emissor functions to load and save emissor JSON
from emissor.persistence import ScenarioStorage
from emissor.persistence.persistence import ScenarioController
from emissor.processing.api import SignalProcessor
from emissor.representation.scenario import Modality, Signal

In [2]:
EMISSOR="./emissor"
SCENARIO="1abc01f0-b1d0-48f9-aafb-60214eaa4380"
#SCENARIO="d5a6bc60-c19b-4c08-aee5-b4dd1c65c64d"

EMISSOR = "/Users/piek/Desktop/d-Leolani/leolani-mmai-parent/cltl-leolani-app/py-app/storage/emissor"
SCENARIO="12f5c2a5-5955-40b2-9e11-45572cd26c75"

### Cleaning any existing annotations

Annotations are added every time you call an annotator. To avoid adding duplicate annotations, the next function removes annotations from a source.
You should call this function before you annotate an emissor JSON to avoid duplicates. Inspect the JSON file to find the source value.

In [4]:
def remove_annotations(signals:[Signal], annotation_source:str):
    for signal in signals:
        keep_mentions = []
        for mention in signal.mentions:
            clear=False
            for annotation in mention.annotations:
                if annotation.source and annotation.source== annotation_source:
                    clear=True
                    break
            if not clear:
                keep_mentions.append(mention)
        signal.mentions = keep_mentions

## Assigning GO Emotions to text signals

In [5]:
model="bhadresh-savani/bert-base-go-emotion"
model_name = "GO"
annotator = EmotionAnnotator(model=model, model_name=model_name)
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

### Assigning MIDAS dialogue acts to text signals

In [6]:
model= "../leolani_text_to_ekg/resources/midas-da-xlmroberta"
model_name="MIDAS"
annotator = DialogueActAnnotator(model=model, model_name=model_name)

scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

You are using a model of type xlm-roberta to instantiate a model of type roberta. This is not supported for all configurations of models and can yield errors.


### Assigning LLM likelihood scores to text signals

In [3]:
model= "../leolani_text_to_ekg/resources/usr-topicalchat-roberta_ft"
model_name="USR"
annotator = LikelihoodAnnotator(model=model, 
                                model_name=model_name, 
                                max_content=300, 
                                top_results=20)

scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
#remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

## Adding spaCy NLP annotations

Depending on the language of the communication, you need to install the proper lanuage model from spaCY.

For English this can be done as follows. Within the same virtual environment, call the next command from the command line in a terminal:

```python -m spacy download en_core_web_sm```



In [9]:
model = 'en_core_web_sm'
model_name ="NLP"
annotator = NLPAnnotator(model=model)
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
### removing any older annotations from this source
remove_annotations(signals, model_name)
### annotating each text signal in the scenario
for signal in signals:
    annotator.process_signal(scenario=scenario_ctrl, text_signal=signal)
#### Save the modified scenario to emissor
scenario_storage.save_scenario(scenario_ctrl)

### Show annotations

In [10]:
scenario_storage = ScenarioStorage(EMISSOR)
scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
signals = scenario_ctrl.get_signals(Modality.TEXT)
for signal in signals:
    mentions = signal.mentions
    for mention in mentions:
        annotations = mention.annotations
        for annotation in annotations:
            print(annotation.type, annotation.value)

ConversationalAgent Ai2Thor
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.GO: 1>, value='neutral', confidence=0.760793924331665)
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.EKMAN: 2>, value='neutral', confidence=0.760793924331665)
python-type:cltl.emotion_extraction.api.Emotion Emotion(type=<EmotionType.SENTIMENT: 4>, value='neutral', confidence=0.7761824033223093)
python-type:cltl.dialogue_act_classification.api.DialogueAct DialogueAct(type='MIDAS', value='command', confidence=3.7178430557250977)
Likelihood 0.3647939035935061
Token Token(text='Hi', pos=<POS.INTJ: 7>, segment=(0, 2))
Token Token(text='Piek', pos=<POS.INTJ: 7>, segment=(3, 7))
Token Token(text='.', pos=<POS.PUNCT: 13>, segment=(7, 8))
Token Token(text='Tell', pos=<POS.VERB: 17>, segment=(9, 13))
Token Token(text='me', pos=<POS.PRON: 11>, segment=(14, 16))
Token Token(text='what', pos=<POS.PRON: 11>, segment=(17, 21))
Token Token(text='to', pos=<POS.PART: 10>, seg

## Annotating all scenarios in EMISSOR

In [3]:
EMISSOR = "/Users/piek/Desktop/t-MA-Combots-2024/assignments/assignment-1/leolani_local/emissor"

## Extract all valid scenarios

In [6]:
import os
folders = os.listdir(EMISSOR)
scenario_folders = []
for scenario in folders:
    if not scenario.startswith("."):
        scenario_path = os.path.join(EMISSOR, scenario)
        has_scenario, has_text, has_image, has_rdf = util.scenario_check.check_scenario_data(scenario_path, scenario)
        check_message = "Scenario:" + scenario + "\n"
        check_message += "\tScenario JSON:" + str(has_scenario) + "\n"
        check_message += "\tText JSON:" + str(has_text) + "\n"
        check_message += "\tImage JSON:" + str(has_image) + "\n"
        check_message += "\tRDF :" + str(has_rdf) + "\n"
        if has_rdf and has_scenario and has_text:
            scenario_folders.append(scenario)
        print(check_message)
print(len(scenario_folders))

Scenario:88b34e91-a5e6-451b-a60e-53c71dc14b95
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:fbad02dc-b38b-4f26-8eb1-6e6e7067fc21
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:f373d946-4549-4d9b-9cfb-be9b64f8d982
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:a589e725-0b5f-4bd2-b945-aa565e750a34
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:3c79d7d6-043c-4344-bda4-6f198c359077
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:5e5a4906-75c7-48ac-9a50-7f5226bb5f45
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:8a083668-f768-4f8a-927d-e706f97b7dcd
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:fe1ebb3a-26c6-4620-b699-1e70c54f91fc
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF :True

Scenario:8bb72942-ae57-433d-831d-1122a9133816
	Scenario JSON:True
	Text JSON:True
	Image JSON:False
	RDF

### Initialise the annotators

In [10]:
#!python -m spacy download en_core_web_sm

In [9]:
model="bhadresh-savani/bert-base-go-emotion"
model_name = "GO"
go_annotator = EmotionAnnotator(model=model, model_name=model_name)

model= "../leolani_text_to_ekg/resources/midas-da-xlmroberta"
model_name="MIDAS"
midas_annotator = DialogueActAnnotator(model=model, model_name=model_name)

model= "../leolani_text_to_ekg/resources/usr-topicalchat-roberta_ft"
model_name="USR"
llh_annotator = LikelihoodAnnotator(model=model, 
                                model_name=model_name, 
                                max_content=300, 
                                top_results=20)

model = 'en_core_web_sm'
model_name ="NLP"
nlp_annotator = NLPAnnotator(model=model)

You are using a model of type xlm-roberta to instantiate a model of type roberta. This is not supported for all configurations of models and can yield errors.


In [11]:
for SCENARIO in scenario_folders:
    print(SCENARIO)
    scenario_storage = ScenarioStorage(EMISSOR)
    scenario_ctrl = scenario_storage.load_scenario(SCENARIO)
    signals = scenario_ctrl.get_signals(Modality.TEXT)
    for signal in signals:
       # go_annotator.process_signal(scenario=scenario_ctrl, signal=signal)
       # midas_annotator.process_signal(scenario=scenario_ctrl, signal=signal)
        llh_annotator.process_signal(scenario=scenario_ctrl, signal=signal)
       # nlp_annotator.process_signal(scenario=scenario_ctrl, text_signal=signal)
    #### Save the modified scenario to emissor
    scenario_storage.save_scenario(scenario_ctrl)

88b34e91-a5e6-451b-a60e-53c71dc14b95
fbad02dc-b38b-4f26-8eb1-6e6e7067fc21
f373d946-4549-4d9b-9cfb-be9b64f8d982
a589e725-0b5f-4bd2-b945-aa565e750a34
3c79d7d6-043c-4344-bda4-6f198c359077
5e5a4906-75c7-48ac-9a50-7f5226bb5f45
8a083668-f768-4f8a-927d-e706f97b7dcd
fe1ebb3a-26c6-4620-b699-1e70c54f91fc
8bb72942-ae57-433d-831d-1122a9133816
a87b3acc-3a80-4516-b182-6eec532e07cb
d5197e01-f2e7-412b-a403-5a8cd7403526
deaefb55-4512-41b3-8008-ace282e1408f
ba5cffd7-d958-4d28-86a7-9f0aae41a8bc


### END